# Unified Quantization Evaluation

End-to-end PPL benchmark covering every configuration from the thesis table.

## What this notebook measures

| Config | Weights | Activations | KV | Hessian? |
|---|---|---|---|---|
| fp16_baseline | FP16 | FP16 | FP16 | — |
| nvfp4_w4a4_nohess | NVFP4 (absmax) | NVFP4 (E4M3 block-16) | FP16 | No |
| w4a16_hadamard | E2M1 Hessian-rotated | FP16 | FP16 | Yes |
| gf4_fixed_w4a4 | E2M1 Hessian-rotated | GF4 fixed-clip | FP16 | Yes |
| gf4_adaptive_w4a4 | E2M1 Hessian-rotated | GF4 per-block adaptive | FP16 | Yes |
| gf4_residual2_w4a4 | E2M1 Hessian-rotated | GF4 2-pass residual | FP16 | Yes |
| nvfp4_acts_hess | E2M1 Hessian-rotated | NVFP4 E4M3 block-16 | FP16 | Yes |
| gf4_fixed_kv4 | E2M1 Hessian-rotated | GF4 fixed-clip | GF4 adaptive | Yes |
| gf4_adaptive_kv4 | E2M1 Hessian-rotated | GF4 per-block adaptive | GF4 adaptive | Yes |
| hw_round_Q14 | NVFP4 | round_Q1.4 lattice | FP16 | No |
| hw_opt_Q14 | NVFP4 | opt_Q1.4 lattice | FP16 | No |
| hw_opt_pop2 | NVFP4 | opt_pop2 lattice | FP16 | No |
| hw_opt_Q14_kv4 | NVFP4 | opt_Q1.4 lattice | GF4 adaptive | No |

**Protocol**: WikiText-2 perplexity, non-overlapping 2048-token windows, GPTQ protocol.
GF4 and hardware-constrained codebooks use **fake-quantization** (codes reconstructed to FP16
before GEMM). Wall-clock / NSIGHT measurements are separate.

**Calibration**: 32 sequences × 2048 tokens from WikiText-2 train split, no gradients.

**Hardware**: A100 80 GB (fits OPT-30B without offload; Hessian path needs CUDA extension).
Models are fully evicted from GPU/CPU after each run to prevent OOM across the suite.

## Setup
1. Ensure `bindings.cpp`, `hadamard_kernel.cu`, `gf4_encode_kernel.cu`,
   `hessian_weight_quant_kernel.cu`, `gf4_fused_gemv_kernel.cu`,
   `e2m1_fused_gemv_kernel.cu` are in the working directory.
2. Set `HF_TOKEN` if you need gated models (LLaMA-2).
3. Run all cells top-to-bottom. Results accumulate in `OUT_CSV`.


In [ ]:
# Cell 1 — Install / auth
!pip install -q transformers datasets accelerate scipy sentencepiece huggingface_hub

HF_TOKEN = ''   # set to 'hf_...' for gated models (LLaMA-2)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN set — open models only.')


In [ ]:
# Cell 2 — Configuration
import os
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# ── Model suite ───────────────────────────────────────────────────────────────
MODEL_SUITE = [
    'facebook/opt-125m',
    'facebook/opt-1.3b',
    'facebook/opt-2.7b',
    'facebook/opt-6.7b',
    'facebook/opt-13b',
    'facebook/opt-30b',
    'meta-llama/Llama-2-7b-hf',
    'meta-llama/Llama-2-13b-hf',
    'meta-llama/Meta-Llama-3-8B',
    'meta-llama/Llama-3.2-1B',
    'meta-llama/Llama-3.2-3B',
    'mistralai/Mistral-7B-v0.1',
    'Qwen/Qwen2.5-7B',
    'Qwen/Qwen2.5-14B',
]

# ── Evaluation protocol ───────────────────────────────────────────────────────
SEQLEN        = 2048
EVAL_WINDOWS  = 10000
CALIB_WINDOWS = 32            # 4 batches × 8 sequences (2048 tok each)

# ── Hadamard rotation mode ────────────────────────────────────────────────────
# True  = full-layer Hadamard: one H of size next_pow2(in_features) per layer.
#         Non-power-of-2 dims (e.g. OPT-2.7B d_model=2560→4096) are zero-padded.
#         Matches original experiments. Spreads all outliers across all features.
# False = blockwise Hadamard: block-32 chunks, mixed independently.
FULL_HADAMARD = True

# ── Quantization hypers ───────────────────────────────────────────────────────
HAD_BLOCK       = 32          # used only when FULL_HADAMARD=False
WBLOCK          = 16          # GF4 and NVFP4 quantization block size
HW_BLOCK        = 32          # hardware-constrained codebook activation block size
HESS_BLOCK      = 32          # Hessian weight-solve block size (≤32, kernel max)
CLIP_RATIO      = 2.5
CLIP_CANDIDATES = (1.5, 2.0, 2.5, 3.0, 4.0)
KV_CLIP_GRID    = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0)
RESIDUAL_PASSES = 2
KAPPA           = 100.0
POWER_ITERS     = 30
KERNEL_DEVICE   = 'cuda:0'
SEED            = 0

# FP16 retention policy — empty: all Linear layers are quantized targets.
# RETAIN_FP16 = ('fc2', 'down_proj', 'lm_head')
RETAIN_FP16 = ()

# ── Output ────────────────────────────────────────────────────────────────────
OUT_CSV     = 'unified_quant_results.csv'
PURGE_CACHE = True

import torch
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.use_deterministic_algorithms(True, warn_only=True)

print(f'CUDA: {torch.cuda.is_available()}  |  '
      f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB'
      if torch.cuda.is_available() else '')
print(f'Suite: {len(MODEL_SUITE)} models  |  calib_windows={CALIB_WINDOWS}  '
      f'eval_windows={EVAL_WINDOWS}')
print(f'FULL_HADAMARD={FULL_HADAMARD}  WBLOCK={WBLOCK}  HW_BLOCK={HW_BLOCK}  RETAIN_FP16={RETAIN_FP16}')


In [ ]:
# Cell 3 — Imports and quantizer primitives
import math, gc, csv, shutil, time, zlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.linalg import hadamard as _scipy_hadamard

# ── Codebook constants ─────────────────────────────────────────────────────────
GF4_LEVEL = np.array(
    [0.0, 0.0796082, 0.1737177, 0.2828685,
     0.3952704, 0.5250730, 0.6961928, 1.0], dtype=np.float32)

E2M1_MAG  = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], dtype=np.float32)

HW_CODEBOOKS = {
    'round_Q1.4': np.array([0,  1,  3,  5,  6,  8, 11, 16], dtype=np.float32) / 16,
    'opt_Q1.4':   np.array([0,  2,  4,  6,  8, 10, 13, 16], dtype=np.float32) / 16,
    'opt_pop2':   np.array([0,  2,  4,  6,  8, 10, 12, 16], dtype=np.float32) / 16,
}

# ── Hadamard rotation ─────────────────────────────────────────────────────────
_H_CACHE = {}
def _hmat(size, device):
    k = (size, str(device))
    if k not in _H_CACHE:
        H = torch.tensor(
            _scipy_hadamard(size).astype(np.float32) / math.sqrt(size),
            device=device)
        _H_CACHE[k] = H
    return _H_CACHE[k]

def _next_pow2(n):
    """Smallest power of 2 >= n."""
    p = 1
    while p < n:
        p <<= 1
    return p

def rotate_pt(x, signs, block=None):
    """
    Hadamard rotation with per-feature sign flip and automatic zero-padding.

    FULL_HADAMARD=True  → _eff_block(K) = next_pow2(K).
      If next_pow2(K) == K  (e.g. 4096): single F×F Hadamard, no padding.
      If next_pow2(K) >  K  (e.g. 2560→4096): zero-pad x/W to 4096, apply
        H_{4096}. Padded cols of both x and W are zero, so the GEMM result
        is identical to the original; the rotation still spreads all K real
        outlier features across all 4096 rotated features.
    FULL_HADAMARD=False → _eff_block(K) = HAD_BLOCK=32.
      Blockwise: independent H_{32} per chunk; F must be divisible by 32.

    block=None is treated as full-layer (eff = F, no padding needed).
    """
    F = x.shape[-1]

    if block is None or block >= F:
        # Full-layer path (possibly with zero-padding when block > F)
        eff = block if (block is not None and block > F) else F
        H   = _hmat(eff, x.device)
        # Apply sign flip only to the K real features
        xs  = x * signs[:F]
        if eff > F:
            # Zero-pad to next power of 2; padded columns contribute 0 to GEMM
            xs = F_.pad(xs, (0, eff - F))
        return xs @ H
    else:
        # Blockwise: F must be divisible by block (holds when FULL_HADAMARD=False)
        xr = (x * signs).reshape(*x.shape[:-1], F // block, block) @ _hmat(block, x.device)
        return xr.reshape(*x.shape[:-1], F)

def _eff_block(K):
    """
    Effective Hadamard size for a layer with in_features=K.
    Full-layer: next_pow2(K) — equals K if already a power of 2, otherwise
    pads (e.g. OPT-2.7B 2560→4096, OPT-125M 768→1024).
    Blockwise: HAD_BLOCK=32.
    """
    return _next_pow2(K) if FULL_HADAMARD else HAD_BLOCK

def _layer_ok(K):
    """
    True if this layer can be rotated under the current FULL_HADAMARD setting.
    Full-layer: always True (any K>0 can be zero-padded to next_pow2(K)).
    Blockwise:  K must be divisible by HAD_BLOCK=32.
    """
    if FULL_HADAMARD:
        return K > 0
    return K % HAD_BLOCK == 0

# Alias so rotate_pt can use torch.nn.functional.pad without shadowing
import torch.nn.functional as F_
# (note: 'F' is also torch.nn.functional above — we only need pad here)

# ── E4M3 scale quantization ────────────────────────────────────────────────────
def quant_e4m3(s):
    s = s.clamp(min=2.0 ** -9, max=448.0)
    e = torch.floor(torch.log2(s))
    m = torch.round((s / torch.exp2(e) - 1.0) * 8.0) / 8.0
    return (1.0 + m) * torch.exp2(e)

# ── GF4 activation quantizer (fixed clip) — block-16 ─────────────────────────
def quant_gf4_fixed(x, clip=CLIP_RATIO):
    shp = x.shape
    xb  = x.reshape(-1, WBLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    sc  = (rms * clip).half().float()
    lv  = torch.tensor(GF4_LEVEL, device=x.device)
    thr = (lv[:-1] + lv[1:]) / 2.0
    idx = torch.bucketize((xb / sc).abs().clamp(0.0, 1.0), thr)
    return (lv[idx] * torch.sign(xb / sc) * sc).reshape(shp).to(x.dtype)

# ── GF4 activation quantizer (per-block adaptive clip) — block-16 ─────────────
def quant_gf4_adaptive(x, candidates=CLIP_CANDIDATES):
    shp = x.shape
    xb  = x.reshape(-1, WBLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    lv  = torch.tensor(GF4_LEVEL, device=x.device)
    thr = (lv[:-1] + lv[1:]) / 2.0
    best_sc  = (rms * candidates[0]).half().float()
    best_mse = torch.full((xb.shape[0], 1), float('inf'), device=x.device)
    for ratio in candidates:
        sc  = (rms * ratio).half().float()
        xn  = xb / sc
        idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
        deq = lv[idx] * torch.sign(xn) * sc
        mse = (deq - xb).pow(2).mean(-1, keepdim=True)
        better   = mse < best_mse
        best_sc  = torch.where(better, sc, best_sc)
        best_mse = torch.where(better, mse, best_mse)
    xn  = xb / best_sc
    idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
    return (lv[idx] * torch.sign(xn) * best_sc).reshape(shp).to(x.dtype)

# ── GF4 KV-cache quantizer (per-block adaptive clip) — block-16 ───────────────
def quant_gf4_kv(x, grid=KV_CLIP_GRID):
    shp = x.shape
    xb  = x.reshape(-1, WBLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    lv  = torch.tensor(GF4_LEVEL, device=x.device)
    thr = (lv[:-1] + lv[1:]) / 2.0
    best_sc  = (rms * grid[0]).half().float()
    best_mse = torch.full((xb.shape[0], 1), float('inf'), device=x.device)
    for ratio in grid:
        sc  = (rms * ratio).half().float()
        xn  = xb / sc
        idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
        deq = lv[idx] * torch.sign(xn) * sc
        mse = (deq - xb).pow(2).mean(-1, keepdim=True)
        better   = mse < best_mse
        best_sc  = torch.where(better, sc, best_sc)
        best_mse = torch.where(better, mse, best_mse)
    xn  = xb / best_sc
    idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
    return (lv[idx] * torch.sign(xn) * best_sc).reshape(shp).to(x.dtype)

# ── NVFP4 weight quantizer — block-16 ────────────────────────────────────────
_E2M1_LV  = None
_E2M1_THR = None
def _ensure_e2m1(device):
    global _E2M1_LV, _E2M1_THR
    if _E2M1_LV is None or _E2M1_LV.device.type != str(device).split(':')[0]:
        _E2M1_LV  = torch.tensor(E2M1_MAG, device=device)
        _E2M1_THR = torch.tensor((E2M1_MAG[:-1] + E2M1_MAG[1:]) / 2.0, device=device)

def nvfp4_weight_quant(W, block=WBLOCK, dev=None):
    """NVFP4 (E2M1) weight quantization at block-16."""
    if dev is None: dev = str(W.device)
    _ensure_e2m1(dev)
    shp = W.shape
    k   = block if shp[1] % block == 0 else (32 if shp[1] % 32 == 0 else 16)
    wb  = W.reshape(-1, k).float()
    amax = wb.abs().amax(1, keepdim=True).clamp_min(1e-8)
    sc   = quant_e4m3(amax / 6.0)
    wn   = wb / sc
    idx  = torch.bucketize(wn.abs().clamp(0.0, 6.0), _E2M1_THR.to(W.device))
    return (_E2M1_LV.to(W.device)[idx] * torch.sign(wn) * sc).reshape(shp).to(W.dtype)

# ── NVFP4 activation quantizer — block-16 ─────────────────────────────────────
def quant_nvfp4_acts(x):
    """NVFP4 (E2M1) activation quantization at block-16."""
    shp  = x.shape
    F    = x.shape[-1]
    xb   = x.float().reshape(-1, F // WBLOCK, WBLOCK)
    amax = xb.abs().amax(-1, keepdim=True).clamp_min(1e-8)
    sc   = quant_e4m3(amax / 6.0)
    xn   = xb / sc
    lv   = torch.tensor(E2M1_MAG, device=x.device)
    thr  = (lv[:-1] + lv[1:]) / 2.0
    idx  = torch.bucketize(xn.abs().clamp(0.0, 6.0), thr)
    return (lv[idx] * torch.sign(xn) * sc).reshape(shp).to(x.dtype)

# ── Hardware-constrained codebook quantizer — block-32 ────────────────────────
def quant_hw_codebook(x, levels_np):
    """Hardware-constrained codebook quantization at block-32 (HW_BLOCK)."""
    shp = x.shape
    xb  = x.reshape(-1, HW_BLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    sc  = (rms * CLIP_RATIO).half().float()
    lv  = torch.tensor(levels_np, device=x.device)
    thr = (lv[:-1] + lv[1:]) / 2.0
    xn  = xb / sc
    idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
    return (lv[idx] * torch.sign(xn) * sc).reshape(shp).to(x.dtype)

print(f'Quantizer primitives loaded.')
print(f'  NVFP4/GF4 block={WBLOCK}  HW codebook block={HW_BLOCK}  Hessian block={HESS_BLOCK}')
print(f'  FULL_HADAMARD={FULL_HADAMARD}  '
      + ('Full F×F Hadamard (padded to next_pow2 for non-PoW2 dims).'
         if FULL_HADAMARD else 'Blockwise 32×32 Hadamard.'))


In [ ]:
# Cell 4 — CUDA extension (Hessian path)
# If the .cu files are not present, the Hessian-path experiments are skipped
# gracefully; all other configs (no-Hessian, hardware-constrained) still run.

from torch.utils.cpp_extension import load as _ext_load

_CUDA_SOURCES = [
    'bindings.cpp',
    'hadamard_kernel.cu',
    'gf4_encode_kernel.cu',
    'e2m1_fused_gemv_kernel.cu',
    'gf4_fused_gemv_kernel.cu',
    'hessian_weight_quant_kernel.cu',
]

ext = None
HESSIAN_AVAILABLE = False

if all(os.path.exists(s) for s in _CUDA_SOURCES):
    print('Loading CUDA extension (reuses cached build if present)...')
    try:
        ext = _ext_load(
            name='gf4_kernels',
            sources=_CUDA_SOURCES,
            extra_cflags=['-O3'],
            extra_cuda_cflags=['-O3', '--use_fast_math'],
            verbose=False,
        )
        HESSIAN_AVAILABLE = True
        print('CUDA extension ready — Hessian path enabled.')
    except Exception as _e:
        print(f'CUDA extension build failed ({_e}). Hessian configs will be skipped.')
else:
    missing = [s for s in _CUDA_SOURCES if not os.path.exists(s)]
    print(f'Missing CUDA source(s): {missing}')
    print('Hessian configs will be skipped; no-Hessian + hardware-constrained still run.')

print(f'HESSIAN_AVAILABLE = {HESSIAN_AVAILABLE}')


In [ ]:
# Cell 5 — WikiText-2 loader + perplexity
from datasets import load_dataset

_WT2_CACHE = {}

def _load_wt2(split):
    if split in _WT2_CACHE:
        return _WT2_CACHE[split]
    for repo in ('Salesforce/wikitext', 'wikitext'):
        try:
            ds = load_dataset(repo, 'wikitext-2-raw-v1', split=split)
            _WT2_CACHE[split] = ds
            return ds
        except Exception:
            pass
    raise RuntimeError('Could not load wikitext-2-raw-v1 — try: pip install -U datasets')

def tokenize_wt2(tok, split):
    ds   = _load_wt2(split)
    text = '\n\n'.join(ds['text'])
    return tok(text, return_tensors='pt').input_ids[0]

def make_windows(ids, seqlen, n):
    avail = ids.numel() // seqlen
    n     = min(n, avail)
    return [ids[i * seqlen:(i + 1) * seqlen].unsqueeze(0) for i in range(n)]

def _input_device(model):
    try:    return model.get_input_embeddings().weight.device
    except: return next(model.parameters()).device

@torch.no_grad()
def perplexity(model, windows):
    """GPTQ-protocol non-overlapping-window perplexity."""
    nll = ntok = 0
    dev = _input_device(model)
    for w in windows:
        w   = w.to(dev)
        out = model(w, labels=w)
        n   = w.numel() - 1
        nll  += out.loss.float().item() * n
        ntok += n
    return math.exp(nll / ntok)

print('WikiText-2 loader and perplexity function ready.')


In [ ]:
# Cell 6 — Hessian reconstruction helpers
# (Only used when HESSIAN_AVAILABLE=True)
# rotate_pt is used here instead of ext.hadamard_fwht so that FULL_HADAMARD=True
# (full-layer F×F rotation) works correctly — the CUDA FWHT kernel is block-32 only.

E2M1_CODEBOOKS_T = torch.tensor([
    [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0],
    [0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0,  6.0],
    [0.25, 0.375, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0],
], dtype=torch.float32)

def _unpack_codes(packed):
    lo = (packed & 0xF).to(torch.uint8)
    hi = ((packed >> 4) & 0xF).to(torch.uint8)
    out = torch.empty(packed.shape[0], packed.shape[1] * 2,
                      dtype=torch.uint8, device=packed.device)
    out[:, 0::2] = lo
    out[:, 1::2] = hi
    return out

def dequantize_e2m1(W_codes, W_alpha, W_bias, block_size, kdev):
    tbl  = E2M1_CODEBOOKS_T.to(kdev)
    M    = W_codes.shape[0]
    u8   = _unpack_codes(W_codes.to(kdev))
    nblk = u8.shape[1] // block_size
    u8   = u8.view(M, nblk, block_size)
    sign = torch.where((u8 & 0x8) != 0, -1.0, 1.0)
    mtbl = tbl[W_bias.to(kdev).long()]
    idx  = (u8 & 0x7).long()
    del u8
    mag  = torch.gather(mtbl, 2, idx)
    del idx, mtbl
    W_hat = sign
    W_hat.mul_(mag).mul_(W_alpha.to(kdev).float().unsqueeze(-1))
    return W_hat.reshape(M, nblk * block_size)

def compute_hessian_quant_state(name, module, X_calib, kdev):
    """
    Hessian-weighted E2M1 weight quantization for one layer.
    Rotation uses rotate_pt (pure PyTorch) so FULL_HADAMARD=True (full F×F)
    and FULL_HADAMARD=False (block-32) both work without CUDA kernel size limits.
    """
    K  = module.in_features
    N  = module.out_features
    eb = _eff_block(K)   # K for full-layer, HAD_BLOCK for blockwise

    seed   = zlib.crc32(name.encode('utf-8')) % (2 ** 31)
    gen    = torch.Generator(device='cpu').manual_seed(seed)
    d_sign = (torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1).to(kdev)

    X     = X_calib.to(kdev).float().contiguous()
    W     = module.weight.data.float().to(kdev).contiguous()
    X_had = rotate_pt(X, d_sign, eb)
    del X   # free pre-rotation tensor; X_had is the only large tensor needed
    W_had = rotate_pt(W, d_sign, eb)
    del W

    # Hessian solve on rotated activations/weights (CUDA kernels)
    H    = ext.hessian_accumulate(X_had, HESS_BLOCK)
    Hd   = ext.hessian_damp_blocks(H, KAPPA, POWER_ITERS)
    Wc, Wa, Wb = ext.hessian_weight_solve(W_had, Hd)
    W_hat = dequantize_e2m1(Wc, Wa, Wb, HESS_BLOCK, kdev)
    del W_had, H, Hd, Wc, Wa, Wb

    mu = X_had.mean(0).contiguous()
    bc = (W_hat @ mu).contiguous()

    # Subsample X_had for clip-ratio MSE: 2048 rows is representative and
    # avoids allocating 3 full copies of a large tensor (e.g. fc2 padded to 32768).
    mse_rows = min(X_had.shape[0], 2048)
    Xf   = X_had[:mse_rows].reshape(-1).contiguous()
    del X_had   # free before MSE loop (Xf is a fresh contiguous copy)
    nblk = Xf.numel() // HESS_BLOCK
    mu_t = mu.repeat(mse_rows)
    best_alpha, best_mse = CLIP_CANDIDATES[0], float('inf')
    for alpha in CLIP_CANDIDATES:
        codes_c, sc_c = ext.gf4_encode(Xf, alpha, True, mu)
        x_dec         = ext.gf4_decode(codes_c, sc_c, nblk)
        mse_c         = float(((Xf - mu_t - x_dec) ** 2).mean())
        del x_dec
        if mse_c < best_mse:
            best_mse, best_alpha = mse_c, alpha

    orig_bias = module.bias.data.float().cpu().clone() if module.bias is not None else None

    st = dict(
        K=K, N=N,
        weight_had_q_half=W_hat.half().cpu().contiguous(),
        bias_correction=bc.float().cpu(),
        orig_bias=orig_bias,
        d_sign=d_sign.cpu(),
        had_block=eb,          # full-layer → eb=K; blockwise → eb=32
        mu=mu.cpu(),
        clip_ratio=best_alpha,
    )
    del W_hat, mu, bc, Xf, mu_t
    return st

print('Hessian helpers ready.')
print(f'  Rotation: {"full-layer (F×F per layer)" if FULL_HADAMARD else "blockwise (32×32 blocks)"}')


In [ ]:
# Cell 7 — Forward-patch builders and KV hook utilities

# ── W4A4 patched-forward (Hessian weights + act_fn activation) ────────────────
def make_w4a4_forward(st, act_fn, kdev):
    """
    Returns a forward closure for one layer.

    Mean-centering protocol:
      x_had   = H @ (diag(d) @ x)              # rotated activation
      x_q     = act_fn(x_had - mu)             # quantize MEAN-SUBTRACTED activation
      y       = W_hat @ x_q + bc + bias         # bc = W_hat @ mu cancels the subtraction
              = W_hat @ (x_q + mu) + bias
             ≈ W_hat @ x_had + bias             # bc restores the mean, quantisation noise stays small

    Without mu-subtraction, bc adds W_hat@mu on top of W_hat@x_had → systematic
    bias that accumulates across all layers → PPL explosion.
    """
    K, N  = st['K'], st['N']
    W_cpu = st['weight_had_q_half']   # shape (N, had_block)
    bc    = st['bias_correction'].to(kdev)   # = W_hat @ mu
    ob    = st['orig_bias'].to(kdev) if st['orig_bias'] is not None else None
    ds    = st['d_sign'].to(kdev)
    hb    = st['had_block']
    mu    = st['mu'].to(kdev)         # per-feature mean of rotated calibration activations

    def forward(x):
        od, ot, os = x.device, x.dtype, x.shape
        xf    = x.reshape(-1, K).float().to(kdev).contiguous()
        x_had = rotate_pt(xf, ds, hb)
        x_q   = act_fn(x_had - mu.unsqueeze(0))   # quantise mean-subtracted
        W     = W_cpu.to(kdev, non_blocking=True)
        y     = x_q.half() @ W.t()
        del W
        y = y.float() + bc.unsqueeze(0)   # bc = W_hat@mu restores the mean
        if ob is not None: y = y + ob.unsqueeze(0)
        return y.to(ot).to(od).reshape(*os[:-1], N)
    return forward

def make_residual_gf4_forward(st, n_passes, kdev):
    """N-pass residual GF4 on mean-subtracted activations."""
    K, N  = st['K'], st['N']
    W_cpu = st['weight_had_q_half']
    bc    = st['bias_correction'].to(kdev)
    ob    = st['orig_bias'].to(kdev) if st['orig_bias'] is not None else None
    ds    = st['d_sign'].to(kdev)
    hb    = st['had_block']
    mu    = st['mu'].to(kdev)
    cr    = st['clip_ratio']

    def forward(x):
        od, ot, os = x.device, x.dtype, x.shape
        xf    = x.reshape(-1, K).float().to(kdev).contiguous()
        x_had = rotate_pt(xf, ds, hb)
        x_c   = x_had - mu.unsqueeze(0)   # mean-centred (already correct)
        q = torch.zeros_like(x_had)
        for _ in range(n_passes):
            r  = (x_c - q).reshape(-1).contiguous()
            qr = quant_gf4_fixed(r.reshape(-1, WBLOCK), clip=cr).reshape(x_had.shape)
            q  = q + qr
        W   = W_cpu.to(kdev, non_blocking=True)
        y   = q.half() @ W.t()
        del W
        y = y.float() + bc.unsqueeze(0)
        if ob is not None: y = y + ob.unsqueeze(0)
        return y.to(ot).to(od).reshape(*os[:-1], N)
    return forward

def make_w4a16_hadamard_forward(st, kdev):
    """
    Hessian-rotated weights, FP16 activations (W4A16-Hadamard ablation).
    No mu subtraction or bc needed: activations are FP16, no quantisation noise,
    and the Hessian solve used uncentred X_had so W_hat approximates W_rot directly.
    """
    K, N  = st['K'], st['N']
    W_cpu = st['weight_had_q_half']
    ob    = st['orig_bias'].to(kdev) if st['orig_bias'] is not None else None
    ds    = st['d_sign'].to(kdev)
    hb    = st['had_block']

    def forward(x):
        od, ot, os = x.device, x.dtype, x.shape
        xf    = x.reshape(-1, K).float().to(kdev).contiguous()
        x_had = rotate_pt(xf, ds, hb)
        W     = W_cpu.to(kdev, non_blocking=True)
        y     = (x_had.half() @ W.t()).float()
        del W
        if ob is not None: y = y + ob.unsqueeze(0)
        return y.to(ot).to(od).reshape(*os[:-1], N)
    return forward

# ── No-Hessian NVFP4 W4A4 (forward-replacement, supports padding) ─────────────
def patch_nvfp4_nohess(model, targets, kdev):
    """
    Pre-compute Hadamard-rotated + NVFP4-quantized weights, store in closure.
    No Hessian, no mean correction — NVFP4 is used directly on rotated x.
    Uses _eff_block(K) so full-layer rotation with zero-padding works for non-PoW2 dims.
    Returns (orig_fwd_dict, []) — restore via restore_weights.
    """
    orig_fwd = {}
    gen      = torch.Generator().manual_seed(SEED)

    for name, mod in targets:
        K   = mod.in_features
        N   = mod.out_features
        eb  = _eff_block(K)
        dev = mod.weight.device

        signs = (torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1)

        W = mod.weight.data.float()
        W_had_q = torch.empty(N, eb, dtype=torch.float16, device='cpu')
        with torch.no_grad():
            s_dev = signs.to(dev)
            for i in range(0, N, 1024):
                chunk   = W[i:i+1024].to(dev)
                chunk_r = rotate_pt(chunk, s_dev, eb)
                W_had_q[i:i+1024] = nvfp4_weight_quant(chunk_r, dev=str(dev)).half().cpu()
                del chunk, chunk_r
            torch.cuda.empty_cache()
        del W

        ob = mod.bias.data.clone().cpu() if mod.bias is not None else None

        def _fwd(x, _s=signs, _eb=eb, _K=K, _N=N, _W=W_had_q, _b=ob):
            od, ot, os = x.device, x.dtype, x.shape
            xf  = x.reshape(-1, _K).float()
            xr  = rotate_pt(xf, _s.to(xf.device), _eb)
            xq  = quant_nvfp4_acts(xr).half()
            W_  = _W.to(xq.device, non_blocking=True)
            y   = (xq @ W_.t()).float()
            del W_
            if _b is not None:
                y = y + _b.to(y.device).float()
            return y.to(ot).to(od).reshape(*os[:-1], _N)

        orig_fwd[name] = mod.forward
        mod.forward    = _fwd

    return orig_fwd, []


def restore_weights(targets, orig_w_or_fwd, hooks):
    """Restore mod.forward (callable) or mod.weight.data (tensor), remove hooks."""
    for h in hooks: h.remove()
    for name, mod in targets:
        if name in orig_w_or_fwd:
            val = orig_w_or_fwd[name]
            if callable(val):
                mod.forward = val
            else:
                mod.weight.data.copy_(val.to(mod.weight.device))


# ── No-Hessian hardware-constrained codebook (forward-replacement) ─────────────
def patch_hw_codebook(model, targets, levels_np):
    """
    NVFP4 weights + hardware-constrained activation codebook (block-32).
    Forward-replacement design; zero-padding handles non-PoW2 dims.
    Returns (orig_fwd_dict, []).
    """
    orig_fwd = {}
    gen      = torch.Generator().manual_seed(SEED)

    for name, mod in targets:
        K   = mod.in_features
        N   = mod.out_features
        eb  = _eff_block(K)
        dev = mod.weight.device

        signs = (torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1)

        W = mod.weight.data.float()
        W_had_q = torch.empty(N, eb, dtype=torch.float16, device='cpu')
        with torch.no_grad():
            s_dev = signs.to(dev)
            for i in range(0, N, 1024):
                chunk   = W[i:i+1024].to(dev)
                chunk_r = rotate_pt(chunk, s_dev, eb)
                W_had_q[i:i+1024] = nvfp4_weight_quant(chunk_r, dev=str(dev)).half().cpu()
                del chunk, chunk_r
            torch.cuda.empty_cache()
        del W

        ob  = mod.bias.data.clone().cpu() if mod.bias is not None else None
        _lv = levels_np

        def _fwd(x, _s=signs, _eb=eb, _K=K, _N=N, _W=W_had_q, _b=ob, _lv=_lv):
            od, ot, os = x.device, x.dtype, x.shape
            xf  = x.reshape(-1, _K).float()
            xr  = rotate_pt(xf, _s.to(xf.device), _eb)
            xq  = quant_hw_codebook(xr, _lv).half()
            W_  = _W.to(xq.device, non_blocking=True)
            y   = (xq @ W_.t()).float()
            del W_
            if _b is not None:
                y = y + _b.to(y.device).float()
            return y.to(ot).to(od).reshape(*os[:-1], _N)

        orig_fwd[name] = mod.forward
        mod.forward    = _fwd

    return orig_fwd, []


# ── KV-cache hooks ────────────────────────────────────────────────────────────
_KV_HOOKS = []

def install_kv_hooks(model):
    global _KV_HOOKS
    _KV_HOOKS = []; found = []
    # Qwen2-style models apply k_norm after k_proj (before attention).
    # Hooking k_proj gives pre-norm keys; k_norm then renormalises the
    # quantised values → attention diverges (measured: >80k PPL on Qwen2.5).
    # Fix: detect k_norm and hook its output instead for key quantisation.
    has_k_norm = any(name.split('.')[-1] == 'k_norm'
                     for name, _ in model.named_modules())
    key_target = 'k_norm' if has_k_norm else 'k_proj'
    for name, mod in model.named_modules():
        last = name.split('.')[-1]
        is_key = (last == key_target) and (isinstance(mod, nn.Linear) or has_k_norm)
        is_val = (last == 'v_proj') and isinstance(mod, nn.Linear)
        if is_key or is_val:
            def _post(m, inp, out): return quant_gf4_kv(out)
            _KV_HOOKS.append(mod.register_forward_hook(_post))
            found.append(name)
    if _KV_HOOKS:
        print(f'    KV hooks: {len(_KV_HOOKS)} (e.g. {found[0]}, {found[1]})')
        if has_k_norm:
            print(f'    [Qwen2-style: key hooks on k_norm output (post-norm)]')
    else:
        sample = [n for n, m in model.named_modules() if isinstance(m, nn.Linear)][:4]
        print(f'    KV hooks: 0 found — Linear sample: {sample}')
    return len(_KV_HOOKS)


def remove_kv_hooks():
    global _KV_HOOKS
    for h in _KV_HOOKS: h.remove()
    _KV_HOOKS = []


print('Forward-patch builders ready.')
print('  make_w4a4_forward: quantises (x_had - mu); bc=W_hat@mu restores mean.')
print(f'  Block sizes — GF4/NVFP4 acts: {WBLOCK}  HW codebook: {HW_BLOCK}')


In [ ]:
# Cell 8 — CSV result logger
import csv as _csv_mod

_CSV_COLS = [
    'timestamp', 'model',
    'fp16_baseline',
    'nvfp4_w4a4_nohess',
    'w4a16_hadamard',
    'gf4_fixed_w4a4',
    'gf4_adaptive_w4a4',
    f'gf4_residual{RESIDUAL_PASSES}_w4a4',
    'nvfp4_acts_hess',
    'gf4_fixed_kv4',
    'gf4_adaptive_kv4',
    'hw_round_Q14',
    'hw_opt_Q14',
    'hw_opt_pop2',
    'hw_opt_Q14_kv4',
    'seqlen', 'eval_windows', 'calib_windows',
]

def _append_row(row_dict):
    new_file = not os.path.exists(OUT_CSV)
    with open(OUT_CSV, 'a', newline='') as f:
        w = _csv_mod.DictWriter(f, fieldnames=_CSV_COLS, extrasaction='ignore')
        if new_file: w.writeheader()
        w.writerow(row_dict)

def fmt(ppl, base):
    if ppl is None: return '   —   '
    return f'{ppl:.4f} ({ppl-base:+.4f})'

def print_row(model_name, row):
    base = row.get('fp16_baseline')
    print(f'\n{"─"*72}')
    print(f'  {model_name}')
    print(f'  FP16 baseline              {base:.4f}')
    for k in _CSV_COLS[2:-3]:  # skip fp16_baseline, seqlen/windows at end
        if k == 'fp16_baseline': continue
        if k in row:
            print(f'  {k:<30s} {fmt(row[k], base)}')
    print(f'{"─"*72}')

print(f'CSV logger ready. Output → {OUT_CSV}')


In [ ]:
# Cell 9a — OPT-6.7B crash recovery (5 remaining Hessian configs)
# then continues with all remaining models.
# Run this cell instead of Cell 9 when restarting after a 6.7B crash.
#
# STREAMING CALIBRATION: activations are captured one layer at a time so peak
# RAM = model_weights + one_layer_activations instead of model + all_activations.
# Safe for OPT-30B on 167 GB. Costs ~N extra forward passes during calibration.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer

_LARGE_MODELS = {'facebook/opt-30b', 'Qwen/Qwen2.5-14B'}

def _disk_free_gb():
    try: return shutil.disk_usage('/').free / 1e9
    except: return float('nan')

def _purge_hf(name):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE as base
    except Exception:
        base = os.path.expanduser('~/.cache/huggingface/hub')
    d = os.path.join(base, 'models--' + name.replace('/', '--'))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f'  [purged {d}]')

def _get_targets(model):
    targets = []
    for name, mod in model.named_modules():
        if (isinstance(mod, nn.Linear)
                and (not RETAIN_FP16 or not any(s in name for s in RETAIN_FP16))
                and _layer_ok(mod.in_features)):
            targets.append((name, mod))
    return targets

def _stream_calib_and_solve(model, targets, calib_wins, kdev):
    """
    Streaming calibration + Hessian solve.
    Registers one hook at a time, runs all calib windows, solves immediately,
    frees activations before moving to the next layer.
    Peak extra RAM = one layer's activations (~500 MB for 6.7B, ~940 MB for 30B).
    """
    rot_state = {}
    in_dev = _input_device(model)
    t0 = time.time()
    for i, (nm, md) in enumerate(targets):
        acts = []
        def _cap(mod, inp, _out, _a=acts):
            _a.append(inp[0].detach().reshape(-1, inp[0].shape[-1]).half().cpu())
        h = md.register_forward_hook(_cap)
        with torch.no_grad():
            for w in calib_wins:
                model(w.to(in_dev))
        h.remove()
        X = torch.cat(acts, 0).float()
        del acts
        # Cap rows: FULL_HADAMARD=True zero-pads large in_features
        # (e.g. fc2 in_features=20480 → eb=32768, X_had = N×32768 float32).
        # 8192 rows keeps X_had ≤1 GB; block-32 Hessian only needs 32+ rows/block.
        if X.shape[0] > 8192:
            X = X[:8192].clone()
        rot_state[nm] = compute_hessian_quant_state(nm, md, X, kdev)
        del X
        gc.collect()
        torch.cuda.empty_cache()
        if (i + 1) % 20 == 0 or (i + 1) == len(targets):
            elapsed = time.time() - t0
            print(f'    {i+1}/{len(targets)} layers  [{elapsed:.0f}s]', flush=True)
    print(f'  Streaming calib+hessian done in {time.time()-t0:.1f}s.')
    return rot_state

all_rows = []

# ── PART 1: OPT-6.7B — only the 5 missing Hessian configs ───────────────────
_67B = 'facebook/opt-6.7b'
print(f'\n{"="*72}')
print(f'  RECOVERY: {_67B} — running 5 remaining Hessian configs only')
print(f'{"="*72}', flush=True)

rot_state = {}; orig_fwd = {}; targets = []; _cap_hooks = []

row_67b = {
    'model': _67B, 'seqlen': SEQLEN,
    'eval_windows': EVAL_WINDOWS, 'calib_windows': CALIB_WINDOWS,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'fp16_baseline':     10.8602,
    'nvfp4_w4a4_nohess': 12.4806,
    'hw_round_Q14':      12.4414,
    'hw_opt_Q14':        12.3650,
    'hw_opt_pop2':       12.3376,
    'hw_opt_Q14_kv4':    12.5171,
    'w4a16_hadamard':    12.3914,
    'gf4_fixed_w4a4':    12.9392,
}
ppl_fp16 = row_67b['fp16_baseline']

try:
    tok   = AutoTokenizer.from_pretrained(_67B, use_fast=False)
    model = AutoModelForCausalLM.from_pretrained(
        _67B, torch_dtype=torch.float16, device_map='auto').eval()
    print(f'  Loaded. dtype={next(model.parameters()).dtype}')

    test_ids  = tokenize_wt2(tok, 'test')
    train_ids = tokenize_wt2(tok, 'train')
    eval_wins  = make_windows(test_ids,  SEQLEN, EVAL_WINDOWS)
    calib_wins = make_windows(train_ids, SEQLEN, CALIB_WINDOWS)
    print(f'  eval_wins={len(eval_wins)} calib_wins={len(calib_wins)}')

    targets = _get_targets(model)
    print(f'  Target layers: {len(targets)}')
    kdev = KERNEL_DEVICE

    if HESSIAN_AVAILABLE:
        gc.collect(); torch.cuda.empty_cache()
        print(f'  [streaming calib+hessian] {len(targets)} layers × {len(calib_wins)} windows...')
        rot_state = _stream_calib_and_solve(model, targets, calib_wins, kdev)

        orig_fwd = {}
        def patch_hess_fwd(builder_fn):
            for nm, md in targets:
                orig_fwd[nm] = md.forward
                md.forward = builder_fn(rot_state[nm])
        def restore_hess_fwd():
            for nm, md in targets:
                md.forward = orig_fwd.get(nm, md.forward)

        print('  [gf4_adaptive_w4a4]', end=' ', flush=True)
        patch_hess_fwd(lambda st: make_w4a4_forward(
            st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
        t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
        row_67b['gf4_adaptive_w4a4'] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        res_key = f'gf4_residual{RESIDUAL_PASSES}_w4a4'
        print(f'  [{res_key}]', end=' ', flush=True)
        patch_hess_fwd(lambda st: make_residual_gf4_forward(st, RESIDUAL_PASSES, kdev))
        t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
        row_67b[res_key] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        print('  [nvfp4_acts_hess]', end=' ', flush=True)
        patch_hess_fwd(lambda st: make_w4a4_forward(
            st, lambda x: quant_nvfp4_acts(x), kdev))
        t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
        row_67b['nvfp4_acts_hess'] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        print('  [gf4_fixed_kv4]', end=' ', flush=True)
        patch_hess_fwd(lambda st: make_w4a4_forward(
            st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
        torch.cuda.empty_cache()
        n_kv = install_kv_hooks(model)
        t0 = time.time(); ppl = perplexity(model, eval_wins)
        remove_kv_hooks(); restore_hess_fwd()
        if n_kv > 0:
            row_67b['gf4_fixed_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
        else:
            print('SKIPPED (no KV hooks found)')

        print('  [gf4_adaptive_kv4]', end=' ', flush=True)
        patch_hess_fwd(lambda st: make_w4a4_forward(
            st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
        torch.cuda.empty_cache()
        n_kv = install_kv_hooks(model)
        t0 = time.time(); ppl = perplexity(model, eval_wins)
        remove_kv_hooks(); restore_hess_fwd()
        if n_kv > 0:
            row_67b['gf4_adaptive_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
        else:
            print('SKIPPED (no KV hooks found)')

        rot_state.clear(); orig_fwd.clear()
        gc.collect(); torch.cuda.empty_cache()
    else:
        print('  [Hessian path skipped — CUDA extension not available]')

except Exception as _e:
    print(f'  ERROR: {type(_e).__name__}: {str(_e)[:200]}')
    import traceback; traceback.print_exc()
finally:
    try: del model
    except: pass
    rot_state.clear(); orig_fwd.clear(); targets.clear()
    for h in _cap_hooks:
        try: h.remove()
        except: pass
    _cap_hooks.clear()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    print(f'  [evicted]  VRAM: {torch.cuda.memory_allocated()/1e9:.2f}/'
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
    if PURGE_CACHE: _purge_hf(_67B)

all_rows.append(row_67b)
print_row(_67B, row_67b)
_append_row(row_67b)
print(f'  [saved to {OUT_CSV}]')

# ── PART 2: Remaining models (full loop, streaming calibration) ───────────────
_REMAINING = [
    'facebook/opt-13b',
    'facebook/opt-30b',
    'meta-llama/Llama-2-7b-hf',
    'meta-llama/Llama-2-13b-hf',
    'meta-llama/Meta-Llama-3-8B',
    'meta-llama/Llama-3.2-1B',
    'meta-llama/Llama-3.2-3B',
    'mistralai/Mistral-7B-v0.1',
    'Qwen/Qwen2.5-7B',
    'Qwen/Qwen2.5-14B',
]

for model_name in _REMAINING:
    print(f'\n{"="*72}')
    print(f'  MODEL: {model_name}  (disk {_disk_free_gb():.0f} GB free)')
    print(f'{"="*72}', flush=True)

    rot_state = {}; orig_fwd = {}; targets = []; _cap_hooks = []

    row = {'model': model_name, 'seqlen': SEQLEN,
           'eval_windows': EVAL_WINDOWS, 'calib_windows': CALIB_WINDOWS,
           'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}

    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        kw  = dict(torch_dtype=torch.float16)
        if model_name in _LARGE_MODELS or 'opt-30b' in model_name:
            kw['device_map'] = 'sequential'
            print(f'  [device_map=sequential for {model_name}]')
        else:
            kw['device_map'] = 'auto'
        model = AutoModelForCausalLM.from_pretrained(model_name, **kw).eval()
        print(f'  Loaded. dtype={next(model.parameters()).dtype}')

        test_ids  = tokenize_wt2(tok, 'test')
        train_ids = tokenize_wt2(tok, 'train')
        eval_wins  = make_windows(test_ids,  SEQLEN, EVAL_WINDOWS)
        calib_wins = make_windows(train_ids, SEQLEN, CALIB_WINDOWS)
        print(f'  eval_wins={len(eval_wins)} calib_wins={len(calib_wins)}')

        targets = _get_targets(model)
        print(f'  Target layers: {len(targets)} '
              f'(FULL_HADAMARD={FULL_HADAMARD}, RETAIN_FP16={RETAIN_FP16})')
        kdev = KERNEL_DEVICE

        t0 = time.time()
        ppl_fp16 = perplexity(model, eval_wins)
        row['fp16_baseline'] = ppl_fp16
        print(f'  fp16_baseline            {ppl_fp16:.4f}  [{time.time()-t0:.1f}s]')

        print('  [nvfp4_w4a4_nohess]', end=' ', flush=True)
        orig_w, hooks = patch_nvfp4_nohess(model, targets, kdev)
        t0 = time.time(); ppl = perplexity(model, eval_wins)
        restore_weights(targets, orig_w, hooks); del orig_w, hooks
        torch.cuda.empty_cache()
        row['nvfp4_w4a4_nohess'] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        for cb_name, cb_levels in HW_CODEBOOKS.items():
            key = 'hw_' + cb_name.replace('.', '').replace('/', '')
            print(f'  [{key}]', end=' ', flush=True)
            orig_w, hooks = patch_hw_codebook(model, targets, cb_levels)
            t0 = time.time(); ppl = perplexity(model, eval_wins)
            restore_weights(targets, orig_w, hooks); del orig_w, hooks
            torch.cuda.empty_cache()
            row[key] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        print('  [hw_opt_Q14_kv4]', end=' ', flush=True)
        orig_w, hooks = patch_hw_codebook(model, targets, HW_CODEBOOKS['opt_Q1.4'])
        n_kv = install_kv_hooks(model)
        t0 = time.time(); ppl = perplexity(model, eval_wins)
        remove_kv_hooks(); restore_weights(targets, orig_w, hooks); del orig_w, hooks
        torch.cuda.empty_cache()
        if n_kv > 0 and ppl >= ppl_fp16:
            row['hw_opt_Q14_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
        else:
            print(f'SKIPPED (n_kv={n_kv}, ppl={ppl:.4f})')

        if HESSIAN_AVAILABLE:
            gc.collect(); torch.cuda.empty_cache()
            print(f'  [streaming calib+hessian] {len(targets)} layers × {len(calib_wins)} windows...')
            rot_state = _stream_calib_and_solve(model, targets, calib_wins, kdev)

            orig_fwd = {}
            def patch_hess_fwd(builder_fn):
                for nm, md in targets:
                    orig_fwd[nm] = md.forward
                    md.forward = builder_fn(rot_state[nm])
            def restore_hess_fwd():
                for nm, md in targets:
                    md.forward = orig_fwd.get(nm, md.forward)

            print('  [w4a16_hadamard]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a16_hadamard_forward(st, kdev))
            t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
            row['w4a16_hadamard'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            print('  [gf4_fixed_w4a4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
            t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
            row['gf4_fixed_w4a4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            print('  [gf4_adaptive_w4a4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
            t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
            row['gf4_adaptive_w4a4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            res_key = f'gf4_residual{RESIDUAL_PASSES}_w4a4'
            print(f'  [{res_key}]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_residual_gf4_forward(st, RESIDUAL_PASSES, kdev))
            t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
            row[res_key] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            print('  [nvfp4_acts_hess]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_nvfp4_acts(x), kdev))
            t0 = time.time(); ppl = perplexity(model, eval_wins); restore_hess_fwd()
            row['nvfp4_acts_hess'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            print('  [gf4_fixed_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
            torch.cuda.empty_cache(); n_kv = install_kv_hooks(model)
            t0 = time.time(); ppl = perplexity(model, eval_wins)
            remove_kv_hooks(); restore_hess_fwd()
            if n_kv > 0:
                row['gf4_fixed_kv4'] = ppl
                print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
            else:
                print('SKIPPED (no KV hooks found)')

            print('  [gf4_adaptive_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
            torch.cuda.empty_cache(); n_kv = install_kv_hooks(model)
            t0 = time.time(); ppl = perplexity(model, eval_wins)
            remove_kv_hooks(); restore_hess_fwd()
            if n_kv > 0:
                row['gf4_adaptive_kv4'] = ppl
                print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
            else:
                print('SKIPPED (no KV hooks found)')

            rot_state.clear(); orig_fwd.clear()
            gc.collect(); torch.cuda.empty_cache()
        else:
            print('  [Hessian path skipped — CUDA extension not available]')

    except Exception as _e:
        print(f'  ERROR: {type(_e).__name__}: {str(_e)[:200]}')
        import traceback; traceback.print_exc()

    finally:
        try: del model
        except: pass
        rot_state.clear(); orig_fwd.clear(); targets.clear()
        for h in _cap_hooks:
            try: h.remove()
            except: pass
        _cap_hooks.clear()
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        print(f'  [evicted]  VRAM: {torch.cuda.memory_allocated()/1e9:.2f}/'
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
        if PURGE_CACHE: _purge_hf(model_name)

    all_rows.append(row)
    print_row(model_name, row)
    _append_row(row)
    print(f'  [saved to {OUT_CSV}]')

print(f'\nAll done. Results in {OUT_CSV}')


In [ ]:
# Cell 9b — Qwen2.5 KV4 re-run (fixed k_norm hook placement)
#
# Run after cells 1-8 (config, imports, helpers, forward-builders, csv-util).
# Loads each Qwen2.5 model, re-runs only the three KV4 configs with the
# corrected install_kv_hooks (hooks k_norm output, not k_proj output).
#
# Root cause of original blowup:
#   Qwen2 applies k_norm after k_proj. The old hook captured pre-norm keys;
#   k_norm then renormalised the quantised garbage → attention diverged.
#   Fix is in install_kv_hooks (cell-forward-builders): hook k_norm instead.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, time

_QWEN_RERUN = [
    'Qwen/Qwen2.5-7B',
    'Qwen/Qwen2.5-14B',
]

for model_name in _QWEN_RERUN:
    print(f'\n{"="*72}')
    print(f'  MODEL: {model_name}  [KV4 re-run — k_norm hook fix]')
    print(f'{"="*72}', flush=True)

    rot_state = {}; orig_fwd = {}; targets = []; _cap_hooks = []
    row_kv = {'model': model_name + '_kv_rerun',
               'seqlen': SEQLEN, 'eval_windows': EVAL_WINDOWS,
               'calib_windows': CALIB_WINDOWS,
               'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}

    try:
        kw = dict(torch_dtype=torch.float16)
        if model_name in _LARGE_MODELS or '14B' in model_name:
            kw['device_map'] = 'sequential'
            print(f'  [device_map=sequential]')
        else:
            kw['device_map'] = 'auto'

        tok   = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        model = AutoModelForCausalLM.from_pretrained(model_name, **kw).eval()
        kdev  = KERNEL_DEVICE
        print(f'  Loaded. dtype={next(model.parameters()).dtype}')

        test_ids  = tokenize_wt2(tok, 'test')
        train_ids = tokenize_wt2(tok, 'train')
        eval_wins  = make_windows(test_ids,  SEQLEN, EVAL_WINDOWS)
        calib_wins = make_windows(train_ids, SEQLEN, CALIB_WINDOWS)

        # FP16 baseline (sanity check)
        t0 = time.time()
        ppl_fp16 = perplexity(model, eval_wins)
        row_kv['fp16_baseline'] = ppl_fp16
        print(f'  fp16_baseline  {ppl_fp16:.4f}  [{time.time()-t0:.1f}s]')

        targets = _get_targets(model)
        print(f'  Target layers: {len(targets)}')

        # ── hw_opt_Q14_kv4 (no Hessian, simple) ──────────────────────────────
        print('  [hw_opt_Q14_kv4]', end=' ', flush=True)
        orig_fwd, _ = patch_nvfp4_nohess(model, targets, kdev)
        torch.cuda.empty_cache()
        n_kv = install_kv_hooks(model)   # now hooks k_norm, not k_proj
        t0 = time.time()
        ppl = perplexity(model, eval_wins)
        remove_kv_hooks()
        restore_weights(targets, orig_fwd, [])
        orig_fwd.clear()
        row_kv['hw_opt_Q14_kv4'] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]  KV={n_kv}')

        # ── Hessian calibration (needed for gf4_*_kv4) ───────────────────────
        try:
            import cuda_ext as ext
            print(f'  Streaming calibration ({len(targets)} layers)...', flush=True)
            rot_state = _stream_calib_and_solve(model, targets, calib_wins, kdev)

            def patch_hess_fwd(make_fn):
                for nm, md in targets:
                    if nm in rot_state:
                        orig_fwd[nm] = md.forward
                        md.forward   = make_fn(rot_state[nm])

            def restore_hess_fwd():
                for nm, md in targets:
                    if nm in orig_fwd:
                        md.forward = orig_fwd[nm]
                orig_fwd.clear()

            # gf4_fixed_kv4
            print('  [gf4_fixed_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
            torch.cuda.empty_cache()
            n_kv = install_kv_hooks(model)
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            remove_kv_hooks(); restore_hess_fwd()
            row_kv['gf4_fixed_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]  KV={n_kv}')

            # gf4_adaptive_kv4
            print('  [gf4_adaptive_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
            torch.cuda.empty_cache()
            n_kv = install_kv_hooks(model)
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            remove_kv_hooks(); restore_hess_fwd()
            row_kv['gf4_adaptive_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]  KV={n_kv}')

            rot_state.clear(); orig_fwd.clear()

        except ImportError:
            print('  [Hessian path skipped — CUDA extension not available]')

    except Exception as _e:
        print(f'  ERROR: {type(_e).__name__}: {str(_e)[:200]}')
        import traceback; traceback.print_exc()

    finally:
        try: del model
        except: pass
        rot_state.clear(); orig_fwd.clear(); targets.clear()
        for h in _cap_hooks:
            try: h.remove()
            except: pass
        _cap_hooks.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
        print(f'  [evicted]  VRAM: {torch.cuda.memory_allocated()/1e9:.2f}/'
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
        if PURGE_CACHE: _purge_hf(model_name)

    print(f'  KV re-run results for {model_name}:')
    base = row_kv.get('fp16_baseline', float('nan'))
    for k in ('fp16_baseline', 'hw_opt_Q14_kv4', 'gf4_fixed_kv4', 'gf4_adaptive_kv4'):
        if k in row_kv:
            v = row_kv[k]
            print(f'    {k:<25s}  {v:.4f} ({v-base:+.4f})')

print('\nCell 9b complete.')


In [ ]:
# Cell 9 — Main evaluation loop
# ─────────────────────────────────────────────────────────────────────────────
# Each model is:
#   1. Loaded (device_map='sequential' for OPT-30B)
#   2. Evaluated on all configs
#   3. Fully deleted and GPU cache cleared
#   4. HF cache purged (if PURGE_CACHE=True)
#
# Calibration: 32 × 2048-token windows from WikiText-2 TRAIN split.
# Evaluation:  full WikiText-2 TEST set (caps to available windows).
# RETAIN_FP16 = () → lm_head, fc2, down_proj are all quantized.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer

_LARGE_MODELS = {'facebook/opt-30b', 'Qwen/Qwen2.5-14B'}

def _disk_free_gb():
    try: return shutil.disk_usage('/').free / 1e9
    except: return float('nan')

def _purge_hf(name):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE as base
    except Exception:
        base = os.path.expanduser('~/.cache/huggingface/hub')
    d = os.path.join(base, 'models--' + name.replace('/', '--'))
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f'  [purged {d}]')

def _get_targets(model):
    """
    All Linear layers eligible for Hadamard rotation.
    _layer_ok(K) checks:
      - FULL_HADAMARD=True  → K must be a power of 2 (full F×F matrix)
      - FULL_HADAMARD=False → K must be divisible by HAD_BLOCK=32
    RETAIN_FP16 = () means no retention — fc2/down_proj/lm_head are included.
    """
    targets = []
    for name, mod in model.named_modules():
        if (isinstance(mod, nn.Linear)
                and (not RETAIN_FP16 or not any(s in name for s in RETAIN_FP16))
                and _layer_ok(mod.in_features)):
            targets.append((name, mod))
    return targets


all_rows = []

for model_name in MODEL_SUITE:
    print(f'\n{"="*72}')
    print(f'  MODEL: {model_name}  (disk {_disk_free_gb():.0f} GB free)')
    print(f'{"="*72}', flush=True)

    # Initialise per-model state so finally can always clear them
    rot_state = {}; captured = {}; orig_fwd = {}; targets = []; _cap_hooks = []

    row = {'model': model_name, 'seqlen': SEQLEN,
           'eval_windows': EVAL_WINDOWS, 'calib_windows': CALIB_WINDOWS,
           'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}

    try:
        # ── Load model ────────────────────────────────────────────────────────
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        kw  = dict(torch_dtype=torch.float16)
        if model_name in _LARGE_MODELS or 'opt-30b' in model_name:
            kw['device_map'] = 'sequential'   # block-sequential offload, A100 80GB
            print(f'  [device_map=sequential for {model_name}]')
        else:
            kw['device_map'] = 'auto'
        model = AutoModelForCausalLM.from_pretrained(model_name, **kw).eval()
        print(f'  Loaded. dtype={next(model.parameters()).dtype}')

        # ── WikiText-2 windows ───────────────────────────────────────────────
        test_ids  = tokenize_wt2(tok, 'test')
        train_ids = tokenize_wt2(tok, 'train')
        eval_wins  = make_windows(test_ids,  SEQLEN, EVAL_WINDOWS)
        calib_wins = make_windows(train_ids, SEQLEN, CALIB_WINDOWS)
        print(f'  eval_wins={len(eval_wins)} calib_wins={len(calib_wins)}')

        targets = _get_targets(model)
        print(f'  Target layers: {len(targets)} '
              f'(FULL_HADAMARD={FULL_HADAMARD}, RETAIN_FP16={RETAIN_FP16})')

        kdev = KERNEL_DEVICE

        # ── FP16 baseline ────────────────────────────────────────────────────
        t0 = time.time()
        ppl_fp16 = perplexity(model, eval_wins)
        row['fp16_baseline'] = ppl_fp16
        print(f'  fp16_baseline            {ppl_fp16:.4f}  [{time.time()-t0:.1f}s]')

        # ── NVFP4 W4A4 no-Hessian ────────────────────────────────────────────
        print('  [nvfp4_w4a4_nohess]', end=' ', flush=True)
        orig_w, hooks = patch_nvfp4_nohess(model, targets, kdev)
        t0 = time.time()
        ppl = perplexity(model, eval_wins)
        restore_weights(targets, orig_w, hooks)
        del orig_w, hooks
        torch.cuda.empty_cache()
        row['nvfp4_w4a4_nohess'] = ppl
        print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        # ── Hardware-constrained codebooks (no Hessian) ───────────────────────
        for cb_name, cb_levels in HW_CODEBOOKS.items():
            key = 'hw_' + cb_name.replace('.', '').replace('/', '')
            print(f'  [{key}]', end=' ', flush=True)
            orig_w, hooks = patch_hw_codebook(model, targets, cb_levels)
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_weights(targets, orig_w, hooks)
            del orig_w, hooks
            torch.cuda.empty_cache()
            row[key] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

        # ── Hardware-constrained opt_Q1.4 + KV4 ──────────────────────────────
        print('  [hw_opt_Q14_kv4]', end=' ', flush=True)
        orig_w, hooks = patch_hw_codebook(model, targets, HW_CODEBOOKS['opt_Q1.4'])
        n_kv = install_kv_hooks(model)
        t0 = time.time()
        ppl = perplexity(model, eval_wins)
        remove_kv_hooks()
        restore_weights(targets, orig_w, hooks)
        del orig_w, hooks
        torch.cuda.empty_cache()
        if n_kv > 0 and ppl >= ppl_fp16:
            row['hw_opt_Q14_kv4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
        else:
            print(f'SKIPPED (n_kv={n_kv}, ppl={ppl:.4f})')

        # ── Hessian path (requires CUDA extension) ────────────────────────────
        if HESSIAN_AVAILABLE:
            # Free hw/nohess closures (W_had_q tensors) before calibration —
            # prevents a 3rd weight copy coexisting with module.weight + W_hat.
            gc.collect()
            torch.cuda.empty_cache()

            # -- Capture calibration activations --
            print('  [calib] capturing activations...', end=' ', flush=True)
            captured = {n: [] for n, _ in targets}
            _cap_hooks = []

            def _make_cap(n):
                def _h(mod, inp, _out):
                    captured[n].append(
                        inp[0].detach().reshape(-1, inp[0].shape[-1]).half().cpu())
                return _h

            for nm, md in targets:
                _cap_hooks.append(md.register_forward_hook(_make_cap(nm)))

            in_dev = _input_device(model)
            with torch.no_grad():
                for w in calib_wins:
                    model(w.to(in_dev))
            for h in _cap_hooks: h.remove()
            _cap_hooks = []
            print(f'done ({CALIB_WINDOWS} windows).')

            # -- Hessian weight reconstruction (one layer at a time) --
            print('  [hessian_quant] solving per-layer...', flush=True)
            rot_state = {}
            t_hess = time.time()
            for nm, md in targets:
                X = torch.cat(captured[nm], dim=0).float()
                del captured[nm]
                rot_state[nm] = compute_hessian_quant_state(nm, md, X, kdev)
                del X
                torch.cuda.empty_cache()
            captured.clear()
            print(f'  Hessian solve done in {time.time()-t_hess:.1f}s.')

            orig_fwd = {}
            def patch_hess_fwd(builder_fn):
                for nm, md in targets:
                    orig_fwd[nm] = md.forward
                    md.forward = builder_fn(rot_state[nm])
            def restore_hess_fwd():
                for nm, md in targets:
                    md.forward = orig_fwd.get(nm, md.forward)

            # -- W4A16 Hadamard-rotated --
            print('  [w4a16_hadamard]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a16_hadamard_forward(st, kdev))
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_hess_fwd()
            row['w4a16_hadamard'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            # -- GF4 fixed clip W4A4 --
            print('  [gf4_fixed_w4a4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_hess_fwd()
            row['gf4_fixed_w4a4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            # -- GF4 adaptive (per-block clip) W4A4 --
            print('  [gf4_adaptive_w4a4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_hess_fwd()
            row['gf4_adaptive_w4a4'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            # -- Residual GF4 W4A4 --
            res_key = f'gf4_residual{RESIDUAL_PASSES}_w4a4'
            print(f'  [{res_key}]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_residual_gf4_forward(st, RESIDUAL_PASSES, kdev))
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_hess_fwd()
            row[res_key] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            # -- NVFP4 activations + Hessian weights --
            print('  [nvfp4_acts_hess]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_nvfp4_acts(x), kdev))
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            restore_hess_fwd()
            row['nvfp4_acts_hess'] = ppl
            print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')

            # -- GF4 fixed + KV4 --
            print('  [gf4_fixed_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_fixed(x, clip=st['clip_ratio']), kdev))
            torch.cuda.empty_cache()
            n_kv = install_kv_hooks(model)
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            remove_kv_hooks()
            restore_hess_fwd()
            if n_kv > 0:
                assert ppl >= ppl_fp16, (
                    f'gf4_fixed_kv4 PPL {ppl:.4f} < FP16 {ppl_fp16:.4f} — stale hook.'
                    ' Restart kernel.')
                row['gf4_fixed_kv4'] = ppl
                print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
            else:
                print('SKIPPED (no KV hooks found)')

            # -- Adaptive GF4 + KV4 --
            print('  [gf4_adaptive_kv4]', end=' ', flush=True)
            patch_hess_fwd(lambda st: make_w4a4_forward(
                st, lambda x: quant_gf4_adaptive(x, CLIP_CANDIDATES), kdev))
            torch.cuda.empty_cache()
            n_kv = install_kv_hooks(model)
            t0 = time.time()
            ppl = perplexity(model, eval_wins)
            remove_kv_hooks()
            restore_hess_fwd()
            if n_kv > 0:
                assert ppl >= ppl_fp16, (
                    f'gf4_adaptive_kv4 PPL {ppl:.4f} < FP16 {ppl_fp16:.4f} — stale hook.'
                    ' Restart kernel.')
                row['gf4_adaptive_kv4'] = ppl
                print(f'{ppl:.4f} ({ppl-ppl_fp16:+.4f})  [{time.time()-t0:.1f}s]')
            else:
                print('SKIPPED (no KV hooks found)')

            # Free Hessian state before model eviction
            rot_state.clear()
            orig_fwd.clear()
            gc.collect()
            torch.cuda.empty_cache()

        else:
            print('  [Hessian path skipped — CUDA extension not available]')

    except Exception as _e:
        print(f'  ERROR: {type(_e).__name__}: {str(_e)[:200]}')
        import traceback; traceback.print_exc()

    finally:
        # ── Evict model and all per-model state from GPU + CPU ───────────────
        try: del model
        except Exception: pass
        try: rot_state.clear()
        except Exception: pass
        try: captured.clear()
        except Exception: pass
        try: orig_fwd.clear()
        except Exception: pass
        try: targets.clear()
        except Exception: pass
        for h in _cap_hooks:
            try: h.remove()
            except Exception: pass
        _cap_hooks.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print(f'  [evicted]  VRAM: '
              f'{torch.cuda.memory_allocated()/1e9:.2f}/'
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
        if PURGE_CACHE:
            _purge_hf(model_name)

    # Save row immediately (survives a later crash)
    all_rows.append(row)
    print_row(model_name, row)
    _append_row(row)
    print(f'  [saved to {OUT_CSV}]')

print(f'\nAll done. Results in {OUT_CSV}')


In [ ]:
# Cell 10 — Summary table

SHOW_COLS = [
    'fp16_baseline',
    'nvfp4_w4a4_nohess',
    'w4a16_hadamard',
    'gf4_fixed_w4a4',
    'gf4_adaptive_w4a4',
    f'gf4_residual{RESIDUAL_PASSES}_w4a4',
    'nvfp4_acts_hess',
    'gf4_fixed_kv4',
    'gf4_adaptive_kv4',
    'hw_round_Q14',
    'hw_opt_Q14',
    'hw_opt_pop2',
    'hw_opt_Q14_kv4',
]

# Header
col_w = 14
header = f'{"model":<30}' + ''.join(f'{c[:col_w]:>{col_w}}' for c in SHOW_COLS)
print(header)
print('─' * len(header))

for row in all_rows:
    base = row.get('fp16_baseline', float('nan'))
    line = f"{row['model']:<30}"
    for c in SHOW_COLS:
        if c not in row:
            line += f'{'—':>{col_w}}'
        elif c == 'fp16_baseline':
            line += f"{row[c]:>{col_w}.4f}"
        else:
            delta = row[c] - base
            line += f"{row[c]:>{col_w-7}.4f}({delta:+.3f})"
    print(line)

print(f'\nWritten to: {OUT_CSV}')
print('Note: GPTQ-protocol WikiText-2 PPL, non-overlapping 2048-token windows.')
print('GF4/hw configs = fake-quantization. Wall-clock = separate NSIGHT run.')
